# Usage Model: Who Gets the Ball

Fourth model in the pipeline (play_type → formation → personnel → player_selection → **usage** → result). `player_selection` already decided the 11 players on the field for a simulated play; this step decides who among them gets the ball -- the rusher on a run play, or the targeted receiver on a pass play (including a `None` "no receiver credited" outcome, covering sacks and uncredited incompletions/throwaways).

This is **not a trained classifier** -- a player-identity label space doesn't generalize across rosters/seasons, same reasoning as `player_selection`. Instead it's a leakage-safe weighted-sampling snapshot built from `features/touch_shares.py::build_touch_shares` (raw historical touch counts at 4 progressively coarser buckets) and `modeling/usage/predict.py` (renormalizes over exactly the on-field candidates, with a smoothing scheme tuned separately for rush vs. target -- they turned out to need very different treatment, see below).

**This notebook documents three real bugs found and fixed via backtesting, each changing the design meaningfully:**
1. A week-1 season cutoff is blind to in-season roster changes (a free-agent RB signed that offseason has no recorded history for his new team) -- fixed by using a mid-season cutoff that reflects how this would actually be used in simulation.
2. Treating "no receiver credited" as just another candidate in a flat renormalization badly inflated it, because its raw count reflects *every* historical play in a bucket while the candidate pool is filtered to just 5 current players -- fixed by computing its probability as its own share of the whole bucket, held fixed, with the remaining mass split among candidates.
3. nflverse's depth-chart rank is numbered **per slot** (e.g. separate X/Z/slot receiver spots), not a single ordered list -- confirmed real data where three different WRs are all tagged `depth_team==1` simultaneously. Worse: even after fixing this with a usage-based rank, the *global* marginal-based prior (how skewed a team's full-season target share is) turned out to be a bad predictor of a *specific play's* target, because target selection is close to uniform once you condition on exactly which 5 skill players share the field. Rushing doesn't have this problem -- a starting RB's dominance holds at every level of aggregation.

In [1]:
import numpy as np
import pandas as pd

from fb_models.dataset import (
    load_pbp_dataset, load_participation_dataset, load_depth_charts_dataset,
    load_players_dataset,
)
from fb_models.formations import normalize_offense_formation
from fb_models.personnel_packages import derive_personnel_package
from fb_models.features.touch_shares import build_touch_shares, player_rank_priors
from fb_models.modeling.usage import predict as usage_predict
from fb_models.modeling.usage.predict import compute_rush_probabilities, compute_target_probabilities

In [2]:
# Season-based split, but with a MID-SEASON cutoff rather than week 1:
# touch_shares/rank_priors are built strictly before TEST_WEEK of TEST_SEASON,
# then evaluated against real plays from TEST_WEEK onward. A week-1 cutoff
# was tried first and performed *worse than a uniform baseline* -- it's
# blind to in-season roster changes (e.g. a free-agent RB signed that
# offseason has zero recorded touches for his new team and isn't on its
# depth chart before the season starts, despite clearly being the workhorse
# by then). Cutting off mid-season instead lets the snapshot see that
# season's own already-elapsed weeks, matching how this would actually be
# used to simulate an in-progress or later game.
SEASONS = list(range(2019, 2025))
TEST_SEASON = 2024
TEST_WEEK = 10

In [3]:
pbp = load_pbp_dataset(seasons=SEASONS)
participation = load_participation_dataset(seasons=SEASONS)
depth_charts = load_depth_charts_dataset(seasons=SEASONS)
players = load_players_dataset()

# Leakage-safe: touch_shares/rank_priors only see plays strictly before TEST_WEEK.
touch_shares = build_touch_shares(pbp, participation, as_of_season=TEST_SEASON, as_of_week=TEST_WEEK)
rank_priors = player_rank_priors(depth_charts, touch_shares, as_of_season=TEST_SEASON, as_of_week=TEST_WEEK)

print("rush league top5:", sorted(touch_shares["rush"]["league"].items(), key=lambda x: -x[1])[:5])
print("target None share:", touch_shares["target"]["league"].get(None, 0) / sum(touch_shares["target"]["league"].values()))

rush league top5: [('00-0032764', 1697), ('00-0035700', 1456), ('00-0033897', 1278), ('00-0035685', 1243), ('00-0033045', 1242)]
target None share: 0.09969395922595226


### The depth-chart rank problem

nflverse's `depth_team` field is numbered per formation *slot*, not as a single ordered depth chart. Confirmed on real 2024 data: PHI's depth chart tags three different wide receivers all `depth_team==1` simultaneously, despite wildly different actual target volumes.

In [4]:
names = players.set_index("gsis_id")["display_name"].to_dict()
phi_wrs = {names.get(pid, pid): entry for (team, pid), entry in rank_priors.items() if team == "PHI" and entry[0] == "WR"}
print("PHI WR usage-based ranks (fixed):", dict(list(phi_wrs.items())[:5]))

PHI WR usage-based ranks (fixed): {'A.J. Brown': ('WR', 1), 'DeVonta Smith': ('WR', 2), 'Julio Jones': ('WR', 3), 'Nelson Agholor': ('WR', 4), 'Zach Pascal': ('WR', 5)}


## Build the held-out evaluation set

Uses the **real** on-field roster from each historical play's actual `offense_players` (bucketed by each player's `position_group`), not a freshly-sampled `player_selection` roster -- this keeps the backtest from being confounded by player_selection's own sampling noise.

In [5]:
test_pbp = pbp[
    (pbp["season"] == TEST_SEASON)
    & (pbp["week"] >= TEST_WEEK)
    & (pbp["down"].notna())
    & (pbp["season_type"] == "REG")
    & (pbp["play_type"].isin(["run", "pass"]))
]

part_cols = participation[
    ["nflverse_game_id", "play_id", "offense_personnel", "offense_formation", "offense_players"]
].copy()
part_cols = derive_personnel_package(part_cols)
part_cols = normalize_offense_formation(part_cols)

eval_df = test_pbp[
    ["game_id", "play_id", "posteam", "play_type", "field_position", "rusher_player_id", "receiver_player_id"]
].merge(
    part_cols,
    left_on=["game_id", "play_id"],
    right_on=["nflverse_game_id", "play_id"],
    how="inner",
)
eval_df = eval_df[eval_df["offense_personnel_package"].notna() & eval_df["offense_formation"].notna()]
print(f"held-out plays: {len(eval_df):,}")

position_group = players.set_index("gsis_id")["position_group"].to_dict()
_BUCKET = {"RB": "RB", "FB": "RB", "TE": "TE", "WR": "WR", "QB": "QB"}

def real_on_field(offense_players):
    on_field = {"QB": [], "RB": [], "TE": [], "WR": []}
    for pid in offense_players:
        if pd.isna(pid):
            continue
        group = _BUCKET.get(position_group.get(pid))
        if group:
            on_field[group].append(pid)
    return on_field

held-out plays: 16,502


## Backtest: score against real outcomes and a uniform baseline

Metric is the probability the model assigns to the *real* rusher/target (mean log-loss across the held-out season), plus top-1 hit rate as a secondary, more interpretable metric.

In [6]:
def score(rows):
    rush_ll, rush_hits, rush_n = [], 0, 0
    target_ll, target_hits, target_n = [], 0, 0
    none_pred, none_actual = [], []

    for row in rows:
        on_field = real_on_field(row.offense_players)
        kwargs = dict(
            team=row.posteam, package=row.offense_personnel_package,
            formation=row.offense_formation, field_position=row.field_position,
            on_field=on_field, touch_shares=touch_shares, rank_priors=rank_priors,
        )

        if row.play_type == "run" and row.rusher_player_id in usage_predict._rush_candidates(on_field):
            probs = compute_rush_probabilities(**kwargs)
            p_true = probs.get(row.rusher_player_id, 1e-6)
            rush_ll.append(-np.log(max(p_true, 1e-6)))
            rush_hits += int(max(probs, key=probs.get) == row.rusher_player_id)
            rush_n += 1

        elif row.play_type == "pass":
            probs = compute_target_probabilities(**kwargs)
            true_target = row.receiver_player_id if pd.notna(row.receiver_player_id) else None
            p_true = probs.get(true_target, 1e-6)
            target_ll.append(-np.log(max(p_true, 1e-6)))
            target_hits += int(max(probs, key=probs.get) == true_target)
            target_n += 1
            none_pred.append(probs.get(None, 0))
            none_actual.append(int(true_target is None))

    return {
        "rush_logloss": np.mean(rush_ll), "rush_hit_rate": rush_hits / rush_n, "rush_n": rush_n,
        "target_logloss": np.mean(target_ll), "target_hit_rate": target_hits / target_n, "target_n": target_n,
        "mean_pred_none": np.mean(none_pred), "actual_none_rate": np.mean(none_actual),
    }

sample = eval_df.sample(n=min(6000, len(eval_df)), random_state=42)
rows = list(sample.itertuples(index=False))
results = score(rows)
for k, v in results.items():
    print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")

rush_logloss: 0.699
rush_hit_rate: 0.757
rush_n: 2507
target_logloss: 1.794
target_hit_rate: 0.219
target_n: 3491
mean_pred_none: 0.101
actual_none_rate: 0.108


### Uniform baseline

Every on-field candidate equally likely -- no data needed, log-loss = -log(1/n_candidates).

In [7]:
rush_uniform_ll, target_uniform_ll = [], []
for row in rows:
    on_field = real_on_field(row.offense_players)
    if row.play_type == "run":
        n = len(usage_predict._rush_candidates(on_field))
        if n:
            rush_uniform_ll.append(-np.log(1 / n))
    else:
        n = len(usage_predict._target_candidates(on_field)) + 1
        target_uniform_ll.append(-np.log(1 / n))

print(f"uniform rush log-loss:   {np.mean(rush_uniform_ll):.3f}   (model: {results['rush_logloss']:.3f})")
print(f"uniform target log-loss: {np.mean(target_uniform_ll):.3f}   (model: {results['target_logloss']:.3f})")

uniform rush log-loss:   1.775   (model: 0.699)
uniform target log-loss: 1.788   (model: 1.794)


### Why rush and target need different smoothing strategies

Rushing has a strong signal: a starting RB (or a scrambling QB) dominates real carries, and that signal holds at every level of aggregation -- heavier smoothing toward the group-aware rank prior (`_RUSH_SMOOTHING_ALPHA`) keeps *helping*, all the way up to very large values, because thin situational buckets are noisier than the stable prior.

Targeting is different. Ranking candidates by their *own play's* touch count and checking what rank the real target actually has reveals the true distribution is close to uniform -- most of a team's season-long target concentration (e.g. a true WR1 getting 40%+ of targets) doesn't hold on any *specific* play once you condition on exactly which 5 skill players are sharing the field. A global marginal-based prior (the same shape used for rush) was tried first and scored *worse than uniform* even on its own -- it reflects the skewed season aggregate, not the much flatter thing that's actually realized play to play.

In [8]:
# Empirical: what LOCAL rank (within just this play's own candidates, ranked
# by their own touch count) does the true target actually have?
from collections import Counter

local_rank_dist = Counter()
for row in rows:
    if row.play_type != "pass":
        continue
    on_field = real_on_field(row.offense_players)
    cands = usage_predict._target_candidates(on_field)
    if not cands:
        continue
    true_target = row.receiver_player_id if pd.notna(row.receiver_player_id) else None
    if true_target is None:
        local_rank_dist["None"] += 1
        continue
    bucket = usage_predict._select_bucket(
        touch_shares["target"], team=row.posteam, package=row.offense_personnel_package,
        formation=row.offense_formation, field_position=row.field_position,
    )
    ranked = sorted(cands, key=lambda pid: -bucket.get(pid, 0))
    local_rank_dist[ranked.index(true_target) + 1 if true_target in ranked else "missing"] += 1

total = sum(local_rank_dist.values())
for k, v in sorted(local_rank_dist.items(), key=lambda x: (isinstance(x[0], str), x[0])):
    print(f"{k}: {v} ({v/total:.3f})")

1: 762 (0.218)
2: 716 (0.205)
3: 563 (0.161)
4: 599 (0.172)
5: 469 (0.134)
None: 376 (0.108)
missing: 6 (0.002)


## Final results

In [9]:
print("RUSH")
print(f"  log-loss: {results['rush_logloss']:.3f}  (uniform: {np.mean(rush_uniform_ll):.3f})")
print(f"  hit rate: {results['rush_hit_rate']:.3f}")
print()
print("TARGET")
print(f"  log-loss: {results['target_logloss']:.3f}  (uniform: {np.mean(target_uniform_ll):.3f})")
print(f"  hit rate: {results['target_hit_rate']:.3f}")
print(f"  mean predicted P(None): {results['mean_pred_none']:.3f}  (actual: {results['actual_none_rate']:.3f})")

RUSH
  log-loss: 0.699  (uniform: 1.775)
  hit rate: 0.757

TARGET
  log-loss: 1.794  (uniform: 1.788)
  hit rate: 0.219
  mean predicted P(None): 0.101  (actual: 0.108)


## Example: PHI, 11 personnel, shotgun, normal field position

Sanity check that the tuned model produces a plausible distribution for a real, recognizable situation.

In [10]:
on_field_example = {
    "QB": ["00-0036389"], "RB": ["00-0036919"], "TE": ["00-0034351"],
    "WR": ["00-0037741", "00-0036912", "00-0035208"],
}
rush_probs = compute_rush_probabilities(
    team="PHI", package="11", formation="SHOTGUN", field_position="normal",
    on_field=on_field_example, touch_shares=touch_shares, rank_priors=rank_priors,
)
target_probs = compute_target_probabilities(
    team="PHI", package="11", formation="SHOTGUN", field_position="normal",
    on_field=on_field_example, touch_shares=touch_shares, rank_priors=rank_priors,
)
print("rush probs:", {k: round(v, 3) for k, v in rush_probs.items()})
print("target probs:", {k: round(v, 3) for k, v in target_probs.items()})

rush probs: {'00-0036389': 0.7, '00-0036919': 0.287, '00-0034351': 0.0, '00-0037741': 0.002, '00-0036912': 0.007, '00-0035208': 0.002}
target probs: {None: 0.118, '00-0036912': 0.215, '00-0034351': 0.198, '00-0036919': 0.182, '00-0035208': 0.156, '00-0037741': 0.131}


## Notes

- Unlike the classifiers earlier in the pipeline, there's no joblib artifact to save here -- `touch_shares`/`player_rank_priors` are built live from `FeatureStore` at simulation time (same as `player_package_shares`/`offense_depth_chart`), not a static trained model file.
- `compute_rush_probabilities`/`compute_target_probabilities`'s output (a probability distribution over on-field candidates, with `None` for target) is what the simulation samples from -- exactly what the not-yet-built result model will condition on next.
- The target model's ceiling (~1.77 log-loss, barely better than uniform's ~1.79) is a genuine property of the data, not a modeling shortfall: once you know which 5 skill players share the field for a personnel grouping, historical season-long share tells you very little about who gets thrown to on any one specific play. Real predictive lift here would need per-play signals this dataset doesn't have (coverage, route concept, pre-snap motion) -- out of scope for this step.